# **Step 1: Environment Setup & Installations.**

In a new cell in your Google Colab notebook, we will first need to install the packages that are not available by default.
Run the following code block to install Gradio (for the web UI) and Sentence-Transformers (for generating the text embeddings):

In [ ]:
# Install the required libraries
!pip install -q gradio sentence-transformers

In [ ]:
# Import necessary libraries
import pandas as pd
import re
import gradio as gr
import torch
from sentence_transformers import SentenceTransformer, util

# Set pandas display options (helps with viewing long text in Colab)
pd.set_option('display.max_colwidth', None)

pandas: For loading and manipulating your financial_news.csv dataset.

re: Python's built-in regular expression library, which we'll use to find and extract the URLs from the text.

gradio: For building the interactive web interface.

SentenceTransformer & util: For loading the embedding model and calculating the cosine similarity efficiently.

torch: The underlying framework used by sentence-transformers.

# **Step 2: Data Loading & Preprocessing.**

In this step, we will load the dataset, isolate the URLs at the end of each text string, move them into their own column, and clean up the original text.

In [ ]:
# Load the dataset (Make sure 'financial_news.csv' is uploaded to your Colab session)
df = pd.read_csv('/content/financial_news.csv')

# 1. Extract the URL from the 'text' column and create the new 'URL' column
# The regex r'(https?://[^\s]+)' looks for 'http://' or 'https://' followed by non-whitespace characters
df['URL'] = df['text'].str.extract(r'(https?://[^\s]+)')

# 2. Remove the URL from the 'text' column
df['text'] = df['text'].str.replace(r'https?://[^\s]+', '', regex=True)

# 3. Clean up any trailing spaces, punctuation, or quotation marks left behind
df['text'] = df['text'].str.strip(' ",\r\n')

# Display the first few rows to verify it worked correctly
df.head()

,text,label,URL
0,"Here are Thursday's biggest analyst calls: Apple, Amazon, Tesla, Palantir, DocuSign, Exxon &amp; more",0,https://t.co/QPN8Gwl7Uh
1,"Buy Las Vegas Sands as travel to Singapore builds, Wells Fargo says",0,https://t.co/fLS2w57iCz
2,"Piper Sandler downgrades DocuSign to sell, citing elevated risks amid CEO transition",0,https://t.co/1EmtywmYpr
3,"Analysts react to Tesla's latest earnings, break down what's next for electric car maker",0,https://t.co/kwhoE6W06u
4,"Netflix and its peers are set for a ‘return to growth,’ analysts say, giving one stock 120% upside",0,https://t.co/jPpdl0D9s4


# **Step 3: Embedding Generation.**

Convert the cleaned text of your financial news into "embeddings." Embeddings are numerical representations (dense vectors) of text that capture semantic meaning, allowing computers to understand the context of sentences rather than just matching exact keywords.

In [ ]:
# 1. Initialize the Sentence Transformer model
# We use 'all-MiniLM-L6-v2' because it is lightweight, fast, and excellent for semantic search tasks.
print("Loading the Sentence Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Generate embeddings for the 'text' column
print("Encoding text into embeddings... (This might take a moment depending on the dataset size)")

# Convert the pandas text column into a python list, then encode.
# Setting convert_to_tensor=True allows us to utilize PyTorch tensors for rapid similarity calculations later.
corpus_embeddings = model.encode(df['text'].tolist(), convert_to_tensor=True, show_progress_bar=True)

print("Embeddings generated successfully!")
print(f"Number of encoded sentences: {corpus_embeddings.shape[0]}")

Loading the Sentence Transformer model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding text into embeddings... (This might take a moment depending on the dataset size)


Batches:   0%|          | 0/531 [00:00<?, ?it/s]

Embeddings generated successfully!
Number of encoded sentences: 16990


**SentenceTransformer('all-MiniLM-L6-v2'):** Downloads and loads a pre-trained AI model from Hugging Face.

**model.encode(...):** Passes every row of your cleaned text column through the neural network.

**convert_to_tensor=True:** Keeps the resulting vectors in PyTorch tensor format, which is heavily optimized for the Cosine Similarity math and we will do in the next step.

**show_progress_bar=True:** Gives you a nice visual loading bar in Colab so you can see how fast it's processing!

# **Step 4: Search Engine Logic (Backend).**

Create the core search function. When a user types a query (like "earnings surprise"), this function will convert that query into an embedding and use Cosine Similarity to compare it against all the news embeddings we generated in Step 3.

Cosine similarity measures the angle between two vectors in a multi-dimensional space. The closer the angle is to 0, the more semantically similar the sentences are!

In [ ]:
# Step 4: Search Engine Logic (Backend)

def search_financial_news(query_text, top_k=5):
    """
    Takes a user query, encodes it, finds the top_k most similar records
    in the dataset using Cosine Similarity, and returns a formatted DataFrame.
    """
    # 1. Encode the user's query into a tensor
    query_embedding = model.encode(query_text, convert_to_tensor=True)

    # 2. Compute cosine similarity between the query and all dataset embeddings
    # util.cos_sim returns a matrix, we take the first row [0] since we only have one query
    cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]

    # 3. Find the indices of the top_k highest scores
    # torch.topk returns the top values and their corresponding indices
    top_results = torch.topk(cos_scores, k=top_k)

    # Extract the indices and scores as standard python lists
    top_indices = top_results.indices.cpu().tolist()
    top_scores = top_results.values.cpu().tolist()

    # 4. Retrieve the corresponding rows from the original dataframe
    results = []
    for score, idx in zip(top_scores, top_indices):
        # Fetch the row data
        row = df.iloc[idx]

        # Append formatted dictionary to our results list
        results.append({
            "Similarity Score": f"{score:.4f}",
            "Cleaned Text": row['text'],
            "Label": row['label'],
            "URL": row['URL']
        })

    # Convert the results list of dictionaries into a Pandas DataFrame for clean display
    return pd.DataFrame(results)

# Test it out locally before building the UI!
print("Testing the search function with the query 'earnings surprise':\n")
test_results = search_financial_news("earnings surprise")
display(test_results)

Testing the search function with the query 'earnings surprise':



,Similarity Score,Cleaned Text,Label,URL
0,0.6029,SaaS Companies Earnings Are Coming: What To Expect?. #stocks #trading #business,5,https://t.co/6CxA8r9DdF
1,0.5983,Earnings season has begun! table from @eWhispers,5,https://t.co/UpQJdKM22x
2,0.5731,@equitydd I expect a lot of volatility and wide ranges. Earnings reporting could pose some challenges.,5,NaN
3,0.5675,Wall Street Breakfast: Earnings Evaluation. #markets #investing #stocks,5,https://t.co/b8AD5CBj9s
4,0.5658,Finding the next surprising earnings winner like Netflix where expectations got too negative,5,https://t.co/n1r0i7M0Fo


**model.encode:** Translates the user's typed search query into the same vector space as your dataset.

**util.cos_sim:** Calculates the similarity score (from -1.0 to 1.0) between the query and every single row in your financial news dataset.

**torch.topk:** Efficiently sorts and grabs just the top 5 highest-scoring matches.

**df.iloc:** Pulls the original text, label, and extracted URL from your DataFrame using the matching index.

# **Step 5: wrap the powerful new backend search function inside a sleek web interface using Gradio.**

Gradio makes it incredibly easy to create a frontend without writing any HTML or CSS. We will define an input textbox for the user's query and a Dataframe component to display our Pandas output.

In [ ]:
# Define the Gradio interface
demo = gr.Interface(
    fn=search_financial_news,            # The search function we built in Step 4
    inputs=gr.Textbox(
        lines=1,
        placeholder="e.g., earnings surprise, regulatory fine...",
        label="Search Financial News"
    ),
    outputs=gr.Dataframe(
        headers=["Similarity Score", "Cleaned Text", "Label", "URL"],
        label="Top 5 Most Relevant Records"
    ),
    title="Financial News Semantic Search",
    description="Enter a financial query to find the most semantically relevant news records from the dataset using Sentence Transformers."
)

# Launch the app!
# share=True creates a temporary public link you can access outside of Colab
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://34b8778aa915dfe3d3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://34b8778aa915dfe3d3.gradio.live
